In [ ]:
# ==============================================================================
# WORLD RELIGION DATABASE (WRD) INGESTION
# 
# ARCHITECTURE NOTE: WRD identity categories are manually extracted into a local 
# CSV. This script uses Strategy A (Local Bulk Parsing) to map them to the schema.
#
# SSSOM ALIGNMENT STRATEGY: 
# 1. ID Synthesis: Follows the standard hierarchical pattern for non-LOD sources. 
#    Roots = 1, 2; Children = 1A, 1B.
# 2. Hierarchy: Built dynamically by mapping Parent_Primary_Label to paths.
# 3. Concept Type: All nodes are assigned skos:Concept.
# ==============================================================================

import os
import sys
import pandas as pd
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "WRD"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="World Religion Database",
    fallback_uri="" # Synthetic CURIEs, no base URI
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
input_csv_file = os.path.join(external_data_dir, "WRD_manual.csv")

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    print(f"[*] Overwriting previous {os.path.basename(raw_ingest_file)} due to local bulk extraction.")
    os.remove(raw_ingest_file)

if not os.path.exists(input_csv_file):
    print(f"[!] Critical Error: Could not find WRD manual extract at {input_csv_file}")
    sys.exit(1)

# --- 4. Load Data & Clean Text ---
print(f"[*] Loading Document: {os.path.basename(input_csv_file)}...")

# Handle the classic Windows smart-quote encoding issue
try:
    df_input = pd.read_csv(input_csv_file, dtype=str, encoding='utf-8-sig')
except UnicodeDecodeError:
    print("[*] UTF-8 decoding failed. Falling back to cp1252 (Windows) encoding...")
    df_input = pd.read_csv(input_csv_file, dtype=str, encoding='cp1252')

# Strip invisible whitespace from headers
df_input.columns = df_input.columns.str.strip()

df_input.fillna("", inplace=True)

# Clean up unicode replacement characters and aggressive whitespace stripping
def clean_text(text):
    if not isinstance(text, str): return ""
    # \ufffd targets the specific '' character, replacing it with a standard apostrophe
    text = text.replace('\ufffd', "'").replace('\x92', "'").replace('’', "'")
    return text.strip()

df_input['Primary_Label'] = df_input['Primary_Label'].apply(clean_text)

# Ensure optional columns exist before cleaning
if 'Parent_Primary_Label' in df_input.columns:
    df_input['Parent_Primary_Label'] = df_input['Parent_Primary_Label'].apply(clean_text)
else:
    df_input['Parent_Primary_Label'] = ""
    
if 'Description' in df_input.columns:
    df_input['Description'] = df_input['Description'].apply(clean_text)
else:
    df_input['Description'] = ""

# --- 5. Synthesize Identifiers and Hierarchy ---
print("[*] Synthesizing hierarchical IDs and mapping ancestry...")

# State trackers
label_to_id = {}
label_to_path = {}
root_counter = 1
child_counters = {} # Tracks how many children a parent has (to assign A, B, C)

unprocessed = df_input.to_dict('records')
processed_rows = []

while unprocessed:
    progress_made = False
    remaining = []
    
    for row in unprocessed:
        label = row['Primary_Label']
        parent_label = row['Parent_Primary_Label']
        desc = row['Description']
        
        # Skip completely empty rows
        if not label:
            continue
            
        # Condition 1: It is a root concept
        if not parent_label:
            cid = str(root_counter)
            root_counter += 1
            
            label_to_id[label] = cid
            label_to_path[label] = label
            child_counters[label] = 0
            
            processed_rows.append({
                'Primary_Label': label,
                'Concept_ID': cid,
                'Parent_IDs': "",
                'Hierarchy_Path': label,
                'Description': desc,
                'Concept_Type': 'skos:Concept'  # Forced to skos:Concept
            })
            progress_made = True
            
        # Condition 2: It is a child concept, and its parent has already been processed
        elif parent_label in label_to_id:
            parent_id = label_to_id[parent_label]
            parent_path = label_to_path[parent_label]
            
            # Generate alphabetical suffix (0 -> A, 1 -> B)
            suffix_idx = child_counters[parent_label]
            suffix_char = chr(65 + suffix_idx)
            child_counters[parent_label] += 1
            
            cid = f"{parent_id}{suffix_char}"
            path = f"{parent_path} > {label}"
            
            label_to_id[label] = cid
            label_to_path[label] = path
            
            # Initialize counter for this node in case it has its own children
            child_counters[label] = 0 
            
            processed_rows.append({
                'Primary_Label': label,
                'Concept_ID': cid,
                'Parent_IDs': parent_id,
                'Hierarchy_Path': path,
                'Description': desc,
                'Concept_Type': 'skos:Concept'  # Forced to skos:Concept
            })
            progress_made = True
            
        # Condition 3: Parent not yet processed, save for next iteration
        else:
            remaining.append(row)
            
    if not progress_made and remaining:
        print("[!] Warning: Found orphaned concepts where the Parent_Primary_Label does not exist in the dataset.")
        print(f"Orphaned labels: {[r['Primary_Label'] for r in remaining]}")
        break
        
    unprocessed = remaining

# --- 6. Output to Bronze Layer Schema ---
print("[*] Formatting to standard 15-column schema...")
total_processed = 0

for row in processed_rows:
    extracted_data = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': row['Primary_Label'],
        'CURIE': f"{SOURCE_PREFIX}:{row['Concept_ID']}",
        'Formal_Label': "",
        'Concept_Type': row['Concept_Type'],
        'Hierarchy_Path': row['Hierarchy_Path'],
        'Synonyms': "",
        'Description': row['Description'],
        'Parent_IDs': row['Parent_IDs'],
        'Concept_ID': row['Concept_ID'],
        'URI': "",
        'Has_Translation': "",
        'Status': "active",
        'Crosswalks': ""
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    total_processed += 1

print(f"\n[COMPLETE] Successfully mapped {total_processed} WRD concepts to Bronze Layer.")